# Lab 05 extra: How to compact a long conversation, and what the summary loses

**Going further, for Lab 05**

## Goal

Skills keep the system prompt small. The conversation still grows: every question, every answer and every skill
the agent loaded stays in the message list and is sent again on each call. **Compaction** replaces the old
messages with a short summary written by a model, and the conversation goes on from the summary. Here we compact
a morning at the Chinook support desk, count what that saves and what the summary call costs, and check what
was lost. The invoice numbers survive the summary. The skills the agent had loaded do not, and it loads them
again.

## What you will learn

- How to see a conversation's input grow from call to call.
- How to compact a conversation in your own code: the messages as text, a summary call, a new message list.
- How to check what a summary kept, with a question that only the old messages can answer.
- How to count the whole cost of compaction, the summary call included.
- Which messages have to stay as they are so that an agent does not load its skills again.

## Setup

The same first cell as every lab: it reads the model name from `models.env` and takes the key from `.env` without
printing it.

In [1]:
from pathlib import Path
import json
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import OpenAI, omit

client = OpenAI(timeout=60, max_retries=2)

## The desk agent and its skills

This is the agent of notebook 2, restated so that this notebook runs on its own. The shop's rules are three Agent
Skills in `skills/`. Each is a folder with a `SKILL.md`: a `name` and a `description` at the top, the steps below
them. The agent's system prompt lists the three names and descriptions, and two tools load the rest:

- `load_skill(name)` returns the steps of one skill;
- `read_skill_file(skill, path)` returns a file inside a skill's folder, when the steps point to one.

The next cell restates all of it, in this order:

1. **The skills and the two functions.** Every `SKILL.md` under `skills/` is read into `SKILLS`: `split("---", 2)`
   cuts off the frontmatter and `yaml.safe_load` reads it. Then `load_skill` and `read_skill_file`, which refuses
   any path that leaves the skill's folder. `FUNCTIONS` maps each tool name to its function.
2. **The two tool schemas, `TOOLS`.** `name` is what the model writes to ask for the tool, and `description` says
   what it does. `parameters` lists the arguments, `required` names those that must be present, and `enum`
   limits a skill name to the three skills. `additionalProperties: False` and `strict: True` make the model's
   arguments match the schema exactly.
3. **The system prompt, `SYSTEM`.** Who the agent helps, how to use the two tools, and the three lines of the
   skill list. The cell prints it.

In [3]:
import yaml

SKILLS_DIR = ROOT / "05_Context_And_Skills" / "skills"

SKILLS = {}
for path in sorted(SKILLS_DIR.glob("*/SKILL.md")):
    before, frontmatter, body = path.read_text().split("---", 2)
    meta = yaml.safe_load(frontmatter)
    SKILLS[meta["name"]] = {"description": meta["description"], "body": body.strip(), "folder": path.parent}


def load_skill(name):
    # The steps of one skill
    return SKILLS[name]["body"]


def read_skill_file(skill, path):
    # One file inside the skill's folder, and nothing outside it
    folder = SKILLS[skill]["folder"].resolve()
    target = (folder / path).resolve()
    if folder not in target.parents or not target.is_file():
        return "No such file in this skill."
    return target.read_text()


FUNCTIONS = {"load_skill": load_skill, "read_skill_file": read_skill_file}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "load_skill",
            "description": "Load the full steps of one skill from the skill list.",
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string", "enum": list(SKILLS)}},
                "required": ["name"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_skill_file",
            "description": "Read a file inside a skill's folder, when the skill's steps point to it.",
            "parameters": {
                "type": "object",
                "properties": {
                    "skill": {"type": "string", "enum": list(SKILLS)},
                    "path": {"type": "string", "description": "Path inside the skill folder"},
                },
                "required": ["skill", "path"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
]

SYSTEM = (
    "You help the support reps of the Chinook music shop. Answer in two or three sentences. "
    "The shop's rules are kept as skills. Each skill below holds the steps for one kind of request. "
    "Before you answer a question about the shop's rules, load the skill that covers it with load_skill, "
    "and read a file inside it with read_skill_file when its steps tell you to. "
    "If no skill covers the question, say that the rules you have do not cover it.\n\n"
    "Skills:\n" + "\n".join(f"- {name}: {skill['description']}" for name, skill in SKILLS.items())
)

print(SYSTEM)

You help the support reps of the Chinook music shop. Answer in two or three sentences. The shop's rules are kept as skills. Each skill below holds the steps for one kind of request. Before you answer a question about the shop's rules, load the skill that covers it with load_skill, and read a file inside it with read_skill_file when its steps tell you to. If no skill covers the question, say that the rules you have do not cover it.

Skills:
- account-changes: Change the email on a customer's account or delete the account, within the once-a-year email rule, and say what deletion keeps. Use when a customer asks to update their email or login, or to close or delete their account.
- damaged-media: Handle a CD or vinyl record that arrived cracked, warped or will not play, with a free replacement, the photo we need, and when to refund instead. Use when a customer reports physical damage to a disc, record or boxed set.
- refund-requests: Decide whether a Chinook order is refunded, which invoic

The loop is the one from notebook 2, with one difference: it works on a message list **we pass in**, and adds to
it. That is what lets one conversation run over several questions. `run_turn` calls the model, runs the tools it
asks for, and repeats until the model answers. It returns the answer, the tools that were called and the input
tokens of each call.

`reply.model_dump(exclude_none=True)` turns the model's reply into a plain dictionary, so that every message in
the list is a dictionary and we can read the list later.

In [4]:
def run_turn(messages, max_turns=8):
    steps, tokens = [], []

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, reasoning_effort=EFFORT or omit
        )
        reply = response.choices[0].message
        tokens.append(response.usage.prompt_tokens)
        messages.append(reply.model_dump(exclude_none=True))

        if not reply.tool_calls:
            return reply.content, steps, tokens

        for call in reply.tool_calls:
            arguments = json.loads(call.function.arguments)
            steps.append(call.function.name + "(" + ", ".join(arguments.values()) + ")")
            result = FUNCTIONS[call.function.name](**arguments)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return "Stopped: the model was still asking for tools after max_turns calls.", steps, tokens

## A morning at the desk, in one conversation

Three requests from a rep, one after another, in the same list. The first one carries the detail we will ask about
later: two invoice numbers. For each request we print the tools the agent called and the input tokens of its last
call.

In [5]:
REQUESTS = [
    "Ticket 14: a customer was charged twice for the digital album The Essential Miles Davis on 2 November 2013, "
    "9.99 each. The payment system shows invoices 58211 and 58217. What do we do?",
    "Next one: a customer in Germany says her vinyl arrived warped, and it is the second copy of the same title "
    "that arrived warped. What do we offer?",
    "A customer in Hungary wants to change her account email. She changed it once already in March. Can she?",
]

conversation = [{"role": "system", "content": SYSTEM}]

for request in REQUESTS:
    conversation.append({"role": "user", "content": request})
    answer, steps, tokens = run_turn(conversation)
    print(">", request[:60] + "...")
    print("  tools:", steps)
    print("  input tokens of the last call:", tokens[-1])
    print("  ", answer)

> Ticket 14: a customer was charged twice for the digital albu...
  tools: ['load_skill(refund-requests)', 'read_skill_file(refund-requests, references/duplicate-charges.md)']
  input tokens of the last call: 1019
   This is a duplicate charge: refund invoice **58217** (the later invoice) in full for **$9.99**, and attach both invoices **58211 and 58217** to the refund request. The refund goes to the original payment method within 5–10 business days after approval; support will decide within 3 business days of your first reply.


> Next one: a customer in Germany says her vinyl arrived warpe...
  tools: ['load_skill(damaged-media)', 'load_skill(refund-requests)']
  input tokens of the last call: 1582
   Since this is the second warped copy of the same title, offer a refund rather than sending a third copy; we’ll need the invoice number to process it. Please send a photo of the warped vinyl, and she can keep the damaged copy until the refund is handled—no return is needed.


> A customer in Hungary wants to change her account email. She...
  tools: ['load_skill(account-changes)']
  input tokens of the last call: 1856
   No—she can change her account email only once per calendar year, and she has already changed it in March. She can change it again starting January 1; invoices already issued will retain the old email, and downloads remain tied to the email on each invoice.


Each request got the skill it needed:

- **Ticket 14:** the refund skill and then its reference file, and the answer names invoice 58217, \$9.99 and both invoice numbers.
- **The warped vinyl:** the damaged-media skill, and then the refund skill, because the second damaged copy is refunded.
- **The email change:** the account skill.

Now the last line of each request. The last call for ticket 14 sent 1,019 input tokens. The last call of the morning sent 1,856, for a question about an email address. It carried the system prompt, all three requests, every answer and every skill text loaded so far.

What is in the list now:

In [6]:
roles = [message["role"] for message in conversation]
print(len(conversation), "messages:", {role: roles.count(role) for role in ["system", "user", "assistant", "tool"]})
print(sum(len(message.get("content") or "") for message in conversation), "characters of content")

17 messages: {'system': 1, 'user': 3, 'assistant': 8, 'tool': 5}
6693 characters of content


17 messages. Five of them are `tool` messages, the four skill bodies and the reference file, and they hold most of the 6,693 characters. Of the eight `assistant` messages, five asked for those tools and three are answers.

## A question only the old messages can answer

Later the rep comes back to the first case:

> Back to ticket 14: which invoice are we refunding, for how much, and which invoice numbers go on the request?

The right answer has three values, and all three are in the first request: refund invoice **58217** (the later
one), for **9.99**, with **58211** and 58217 on the request. `check` looks for the three values in a reply.

**What the check establishes:** that the reply names the three values. It reads text, so it says nothing about
whether the rest of the reply is right.

In [7]:
CHECK = "Back to ticket 14: which invoice are we refunding, for how much, and which invoice numbers go on the request?"


def check(answer):
    # The three values the reply must name
    for label, value in [("refunds 58217", "58217"), ("for 9.99", "9.99"), ("names 58211 too", "58211")]:
        print("✅" if value in answer else "❌", label)

First with the whole conversation. `conversation + [...]` makes a new list with the check question at the end and
leaves `conversation` as it was.

In [8]:
full = conversation + [{"role": "user", "content": CHECK}]

answer, steps, full_tokens = run_turn(full)
print("tools:", steps)
print("input tokens per call:", full_tokens)
print(answer)
check(answer)

tools: ['load_skill(refund-requests)', 'read_skill_file(refund-requests, references/duplicate-charges.md)']
input tokens per call: [1945, 2196, 2449]
Refund invoice **58217** in full for **$9.99**; it’s the later invoice. Include both **58211** and **58217** on the refund request.
✅ refunds 58217
✅ for 9.99
✅ names 58211 too


The check passes: 58217, 9.99 and both numbers, all of which are in the first request.

Now read `tools`. In our run the agent loaded the refund skill and its reference **again** before answering, although both are already in the list from the first request. Three calls, 1,945, 2,196 and 2,449 input tokens. The system prompt says to load the covering skill before answering a question about the rules, and for this new question the model did. It does not happen on every run: in two runs of six the agent answered from the list in one call.

## Compact the conversation in three steps

### Step 1: the messages as one text

The model that writes the summary needs the conversation as text. `as_text` writes one message as one line: its
role and its content, or, for a message in which the model asked for tools, the tools it asked for.
`conversation[1:]` leaves out the system prompt, which stays as it is.

In [9]:
def as_text(message):
    # One message as text: its role, then its content or the tools it asked for
    if message.get("tool_calls"):
        calls = [call["function"]["name"] + "(" + call["function"]["arguments"] + ")" for call in message["tool_calls"]]
        return "assistant called: " + "; ".join(calls)
    return message["role"] + ": " + (message.get("content") or "")


transcript = "\n\n".join(as_text(message) for message in conversation[1:])

print(len(transcript), "characters")
print(transcript[:600])

5932 characters
user: Ticket 14: a customer was charged twice for the digital album The Essential Miles Davis on 2 November 2013, 9.99 each. The payment system shows invoices 58211 and 58217. What do we do?

assistant called: load_skill({"name":"refund-requests"})

tool: # Refund requests

Follow these steps for any request to give money back.

1. **Find the invoice.** Ask for the invoice number if the customer did not give it. Never guess one.
2. **Check the reason against the rules:**
   - A customer who changed their mind about a digital purchase gets no refund.
   - When only some tracks on a multi-disc s


The transcript starts with the first request, then the agent's call to `load_skill`, then the whole body of the
refund skill as a `tool` line. Most of the transcript is skill text like this.

### Step 2: the summary call

A separate model call with its own instruction: keep every open case with its numbers, leave out tool output, stay
under 80 words. The 80 is our choice for this lab.

In [10]:
SUMMARY_INSTRUCTION = (
    "You compact a support desk conversation so that the assistant can continue it. Write a summary of at most "
    "80 words. Keep every open case with its ticket number, invoice numbers, amounts and the decision taken. "
    "Leave out tool outputs that are no longer needed."
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SUMMARY_INSTRUCTION},
        {"role": "user", "content": transcript},
    ],
    reasoning_effort=EFFORT or omit,
)
summary = response.choices[0].message.content
summary_tokens = response.usage.prompt_tokens

print(summary)
print()
print(len(summary.split()), "words | the summary call read", summary_tokens, "input tokens")

- **Ticket 14:** Duplicate charge for *The Essential Miles Davis* on 2 Nov 2013. Refund later invoice **58217** for **$9.99**; attach invoices **58211 and 58217**. Refund goes to original payment method 5–10 business days after approval; decision within 3 business days of first reply.
- **Warped vinyl (Germany):** Second copy of same title warped. Offer a refund, not a third copy; request a damage photo and invoice number. Customer keeps the copy.
- **Email change (Hungary):** Declined—already changed in March. Eligible again Jan 1; existing invoices retain old email and downloads stay tied to invoice email.

96 words | the summary call read 1421 input tokens


96 words, over the 80 we asked for: a word limit in an instruction is a request, not a cap. Read what the summary kept:

- **Ticket 14:** invoice 58217, \$9.99, and both invoice numbers. The three values of our check are there.
- **The warped vinyl and the email change:** the decision in each case.

And what it left out, as instructed: the tool outputs, which here are the skills' steps. The summary call read 1,421 input tokens, the whole transcript.

### Step 3: the compacted list

The new list has three messages: the system prompt, the summary as a user message, and the check question. The
requests, the answers and the loaded skills are gone.

In [11]:
summary_message = {"role": "user", "content": "Here is a summary of the conversation so far:\n\n" + summary}

compacted = [conversation[0], summary_message, {"role": "user", "content": CHECK}]

answer, steps, compact_tokens = run_turn(compacted)
print("tools:", steps)
print("input tokens per call:", compact_tokens)
print(answer)
check(answer)

tools: ['load_skill(refund-requests)', 'read_skill_file(refund-requests, references/duplicate-charges.md)']
input tokens per call: [654, 905, 1158]
Refund the later invoice, **58217**, in full for **$9.99**. Put both invoices, **58211 and 58217**, on the refund request.
✅ refunds 58217
✅ for 9.99
✅ names 58211 too


The check passes on the compacted list too. The summary carried the three values.

The first call is small, 654 input tokens against 1,945 for the whole list. But read `tools`: the agent loaded `refund-requests` and read its reference before answering, three calls in all. On the compacted list it did this in every run we made. The summary says **what** was decided. The steps the agent is told to follow before it answers are no longer in the list, so it fetched them.

## Count the whole cost

Compaction is two things: the summary call, and the conversation that continues from the summary. We add up the
input tokens of both and set them beside the whole list.

In [12]:
print("whole list:     ", len(full_tokens), "call(s),", sum(full_tokens), "input tokens")
print("summary call:    1 call,", summary_tokens, "input tokens")
print("compacted list: ", len(compact_tokens), "call(s),", sum(compact_tokens), "input tokens")
print("compaction in all:", summary_tokens + sum(compact_tokens), "input tokens")

whole list:      3 call(s), 6590 input tokens
summary call:    1 call, 1421 input tokens
compacted list:  3 call(s), 2717 input tokens
compaction in all: 4138 input tokens


With our numbers:

- **The whole list:** 6,590 input tokens over three calls, because the agent loaded the skill again. When it answers in one call, it is 1,945.
- **Compaction:** 1,421 for the summary call and 2,717 for the three calls after it, 4,138 in all.

Against this run's whole list (6,590), compaction saved about a third. Against a whole list that answers in one
call (1,945, as in two of our six runs), it cost more than twice as much. Either way the summary call and the two
loads are added on top of a first call of 654 input tokens, a third of the whole list's 1,945.

**Important:** count every call a feature makes. A summary is written by a model call that reads the whole
conversation once more, and an agent that has lost what it loaded makes extra calls to get it back.

## Keep the skill loads through compaction

The agent loaded the skill again because the skill's text left with the old messages. Two ways to give it back.

### First try: paste the skill's text into the summary message

In [13]:
pasted_message = {
    "role": "user",
    "content": summary_message["content"]
    + "\n\nSkills already loaded in this conversation:\n\n"
    + load_skill("refund-requests")
    + "\n\n"
    + read_skill_file("refund-requests", "references/duplicate-charges.md"),
}

pasted = [conversation[0], pasted_message, {"role": "user", "content": CHECK}]

answer, steps, pasted_tokens = run_turn(pasted)
print("tools:", steps)
print("input tokens per call:", pasted_tokens)
check(answer)

tools: ['load_skill(refund-requests)', 'read_skill_file(refund-requests, references/duplicate-charges.md)']
input tokens per call: [1094, 1345, 1598]
✅ refunds 58217
✅ for 9.99
✅ names 58211 too


The check passes, and the agent loaded the skill and its reference again all the same: three calls of 1,094, 1,345 and 1,598 input tokens. This is the most expensive version so far. We paid for the skill text in the user message, and then for the same text again as tool results.

### Second try: keep the original messages of each load

When the agent loaded a skill, two messages went into the list: the model's own request (an `assistant` message
with `tool_calls`) and our result (a `tool` message with the same id). We keep those pairs exactly as they are and
compact everything else. In this notebook every tool is a skill tool, so the pairs are easy to pick out: every
message that has `tool_calls`, and every `tool` message.

In [14]:
kept = [message for message in conversation if message.get("tool_calls") or message["role"] == "tool"]

print(len(kept), "messages kept:")
for message in kept:
    print("  ", as_text(message)[:70].replace("\n", " "))

10 messages kept:
   assistant called: load_skill({"name":"refund-requests"})
   tool: # Refund requests  Follow these steps for any request to give mo
   assistant called: read_skill_file({"skill":"refund-requests","path":"r
   tool: # Duplicate charges  A duplicate charge is **two invoices for th
   assistant called: load_skill({"name":"damaged-media"})
   tool: # Damaged media  1. **Ask for a photo** of the damage. Damaged m
   assistant called: load_skill({"name":"refund-requests"})
   tool: # Refund requests  Follow these steps for any request to give mo
   assistant called: load_skill({"name":"account-changes"})
   tool: # Account changes  An account is one email plus one customer rec


In [15]:
with_kept = [conversation[0], *kept, summary_message, {"role": "user", "content": CHECK}]

answer, steps, kept_tokens = run_turn(with_kept)
print("tools:", steps)
print("input tokens per call:", kept_tokens)
print(answer)
check(answer)

tools: []
input tokens per call: [1788]
Refund invoice **58217** in full for **$9.99**. Attach both invoice numbers, **58211 and 58217**, to the refund request.
✅ refunds 58217
✅ for 9.99
✅ names 58211 too


`tools` is empty. One call, 1,788 input tokens, and the check passes. With the original load messages in the list, the agent answered without loading anything.

**Why the pairs work and the pasted text does not.** The system prompt says to load the skill before answering. A
user message that contains the steps is text about the skill. The pair of messages is the record of the load
itself: a request written by the model and a result with the same id, and here it sits right before the question.

**What we cannot claim:** that the pairs alone are enough wherever they sit. The whole list holds the same pairs, several turns back, and there the agent often loaded again. Kept directly before the summary and the question, it did not in any of our runs.

Two rules follow for any compaction step:

- **Keep a request and its results together.** An `assistant` message with `tool_calls` must be followed by a
  `tool` message for each call, or the API refuses the list.
- **With other tools in the agent, choose by name.** Keep the pairs whose tool is `load_skill` or
  `read_skill_file`, and let the summary cover the lookups.

The three ways, on the same check question:

| What follows the system prompt | Calls | Input tokens | Loaded the skill again |
|---|---|---|---|
| The summary | 3 | 2,717 | yes |
| The summary with the skill text pasted in | 3 | 4,037 | yes |
| The kept load messages, then the summary | 1 | 1,788 | no |

Keeping the pairs is the cheapest of the three and the only one without a second load. Notice also how little it removed: 1,788 input tokens against 1,945 for one call on the whole list. Most of this conversation is skill text, and skill text is what we kept. Compaction saves the most on a conversation that is long in talk and lookups, and the least on one that is long in loaded skills.

**When compaction pays.** On a conversation this short there is little to gain. It pays when the
list is long enough to matter, close to the limit of the model's context or long enough that the model starts to
miss what is in it, and when the summary is then reused over many later turns.

**In production:** you rarely write this step yourself. The Responses API can compact on the server
(`context_management` with a `compact_threshold`), LangChain has a `SummarizationMiddleware`, and coding agents
compact their own conversations when they fill up. The guide for programs that support Agent Skills says the
same thing this notebook measured: protect loaded skill content from compaction.

**About the saved outputs.** The counts come from one run on the model `models.env` names: 19 model
calls. The summary's wording changes from run to run, and so do the
token counts that depend on it. What held in every run we made (six of each): the three values survived the summary; the
agent loaded the refund skill again after the summary alone and after the pasted text; and it did not with the
original load messages kept. With the whole list it loaded again in four runs of six.

**Checkpoint.** The last call of the third request sends nearly twice the input tokens of the last call of the
first (ours: 1,856 against 1,019). The summary
names 58217, 9.99 and 58211. On the compacted list the check passes and `tools` shows `load_skill` and
`read_skill_file` again; with the load messages kept, `tools` is empty and the check passes in one call.

## What you can do now

- Show how a conversation's input grows, from the `prompt_tokens` of its calls.
- Compact a conversation: write the messages as text, summarise them with a model call, build the new list.
- Check what a summary kept with a question about the old messages, and say what the check does not cover.
- Count the cost of compaction with the summary call in it.
- Keep the request and result messages of loaded skills through a compaction.

## Resources

- [OpenAI: compaction](https://developers.openai.com/api/docs/guides/compaction): compaction on the server with
  `context_management`
- [Agent Skills: adding skills support to an agent](https://agentskills.io/client-implementation/adding-skills-support):
  the section on protecting skill content from compaction
- [LangChain: built-in middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in):
  `SummarizationMiddleware`
- [OpenAI: function calling](https://developers.openai.com/api/docs/guides/function-calling): tool calls and the
  `tool` messages that answer them

Back to [notebook 1](05a_meet_context_and_skills.ipynb) or
[notebook 2](05b_context_and_skills_at_the_desk.ipynb).